# Part 2 — MYH Curated Applications Dataset (2020–2025)

**Notebook role:** This notebook will become the full, rerunnable raw-to-curated workflow for Part 2 of the Data Pipeline Project.

**Current implementation status:**  
Sub-project 2.1 has established the notebook architecture, project paths, and workspace conventions.  
The later sections are intentionally prepared as structured implementation slots for Sub-projects 2.2–2.7.

## Project purpose

The finished notebook should:
- read the original MYH Excel workbooks for application rounds **2020–2025**,
- build a longitudinal curated applications dataset from **`Tabell 3`**,
- preserve a clear main-table grain: **one row = one application in one application round**,
- document source differences, harmonization choices, cleaning choices, enrichment, and validation,
- export a final dataset that can later be loaded into SQL and served through a read-oriented API.


## 1. Scope and design principles

### Fixed project direction
- Source years: **2020, 2021, 2022, 2023, 2024, 2025**
- Main source sheet: **`Tabell 3`**
- Main table grain: **one application in one application round per row**
- `Tabell 4` is acknowledged as useful but must **not** be merged into the main applications table in a way that duplicates applications.

### Working quality standard
This notebook should read as an explanatory data journey rather than a code dump.  
Every major transformation will later be accompanied by:
1. what is being done,
2. why it is being done,
3. what trade-off or source inconsistency it addresses.


## 2. Imports and runtime setup

This cell contains the small set of general-purpose libraries expected in the notebook.  
Additional imports should only be added later when they support a concrete implementation need.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)


## 3. Project paths and folder conventions

The notebook is designed to work whether VS Code runs it from:
- the repository root, or
- the `part_2/` folder itself.

The raw-vs-processed rule is simple:
- `data/raw/` contains the unchanged MYH Excel inputs,
- `data/processed/` contains notebook-generated exports.


In [ ]:
def resolve_part_2_dir() -> Path:
    """Return the Part 2 workspace when run from repo root or from part_2/."""
    cwd = Path.cwd().resolve()

    if cwd.name == "part_2":
        return cwd

    candidate = cwd / "part_2"
    if candidate.exists():
        return candidate

    raise FileNotFoundError(
        "Could not locate the 'part_2' folder. "
        "Run this notebook from the repository root or from the part_2/ folder."
    )


PART_2_DIR = resolve_part_2_dir()
RAW_DATA_DIR = PART_2_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PART_2_DIR / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Part 2 workspace: {PART_2_DIR}")
print(f"Raw input folder: {RAW_DATA_DIR}")
print(f"Processed export folder: {PROCESSED_DATA_DIR}")


## 4. Raw-data inventory check

This small setup check confirms which Excel files are currently present in `data/raw/`.  
The full workbook and sheet exploration belongs to **Sub-project 2.2**, but this inventory makes the workspace immediately usable.


In [ ]:
EXPECTED_SOURCE_YEARS = [2020, 2021, 2022, 2023, 2024, 2025]

raw_excel_files = sorted(RAW_DATA_DIR.glob("*.xlsx"))

print(f"Excel workbooks currently found: {len(raw_excel_files)}")
for file_path in raw_excel_files:
    print(f"- {file_path.name}")

if not raw_excel_files:
    print(
        "\nNo raw Excel files have been found yet. "
        "Place the six original MYH 2020–2025 workbooks in data/raw/ "
        "before starting Sub-project 2.2."
    )


## 5. Source-file understanding *(to be implemented in Sub-project 2.2)*

This section will provide notebook-ready evidence that the source files are understood before transformation begins.

Planned content:
- workbook and sheet inventory,
- confirmation that the relevant table is `Tabell 3`,
- explanation of why `Tabell 4` has a different grain,
- row-count and column-count summaries by year,
- header-row differences across years,
- first schema comparison across 2020–2025.


## 6. Why `Tabell 3` is the main source and `Tabell 4` is not merged

This section will be completed with concrete evidence in Sub-project 2.2.

The intended argument is:
- `Tabell 3` supports a main applications table at the correct grain,
- `Tabell 4` contains repeated application identifiers because it operates at a more detailed level,
- blindly merging `Tabell 4` into the main table would risk duplicated applications and misleading counts.


## 7. Initial curated-table design outline *(to be finalized in Sub-project 2.3)*

The final curated schema is not locked in Sub-project 2.1.  
However, the project already has a clear design direction:

### Required characteristics
- stable, SQL/API-friendly field names,
- one row per application per application round,
- traceability fields such as `source_year`, `source_file`, `source_sheet`, and preferably `source_row`,
- raw source values retained where helpful,
- normalized companion fields added where cross-year comparison requires harmonization.

### Examples of already anticipated normalization
- source decision values → `beslut_normalized`,
- source provider-type values → `huvudmannatyp_normalized`.

The formal source-to-target mapping table and final column order will be settled later.


## 8. Reusable ingestion and standardization *(Sub-project 2.4)*

This section will later contain:
- explicit file metadata/configuration,
- year-specific `header=` handling for `Tabell 3`,
- reusable reading logic,
- harmonized column names,
- concatenation into a combined longitudinal base table.


## 9. Cleaning, normalization, and enrichment *(Sub-project 2.5)*

This section will later contain:
- datatype conversions,
- safe text cleanup,
- normalized decision categories,
- normalized provider-type categories,
- selected useful derived fields,
- checks confirming the transformations are safe and complete.


## 10. Validation and quality checks *(Sub-project 2.6)*

This section will later implement stronger-than-minimal confidence checks, such as:
- row counts by source year,
- missing identifier checks,
- duplicate `diarienummer` checks within each year,
- normalized-value mapping coverage,
- missingness review,
- datatype checks,
- sanity summaries by year and category.


## 11. Export of the curated dataset *(Sub-project 2.6)*

The final curated dataset will be written into:

```text
part_2/data/processed/
```

CSV is the expected baseline export.  
Whether a Parquet export adds enough value to keep will be decided later, not prematurely.


## 12. SQL/API handoff note and final reflection *(Sub-project 2.7)*

This final section will explain:
- what table was exported,
- its grain and intended downstream use,
- how it supports later SQL loading,
- how it can support simple read-oriented FastAPI endpoints,
- what limitations or future extensions remain.


## Sub-project 2.1 checkpoint

The architecture phase is complete when:
- this notebook skeleton exists in `part_2/main.ipynb`,
- the raw/processed folder strategy is established,
- the notebook section sequence is stable,
- the project control files hand off cleanly into Sub-project 2.2.
